# Logistic Regression — Play Golf Prediction
# Binary Classification with Scikit-Learn

---

**Repository:** ML with Scikit-Learn  
**Notebook:** 02 — Golf Play Prediction  
**Algorithm:** Logistic Regression  
**Encoding:** One-Hot Encoding via `pd.get_dummies`  
**Dataset:** Golf Play Dataset (categorical features)  
**Author:** Ather-ops  

---

## Objective

Predict whether someone will **play golf** based on weather conditions — Temperature, Humidity, and Windy. This is a classic binary classification problem with entirely categorical input features, which requires encoding before any ML model can process them.

---

## What is Different About This Dataset

All previous classification notebooks (student scores, fraud detection) had numerical features. This dataset has **only categorical text columns** — `Hot`, `High`, `Yes`, `No`. Machine learning models cannot process text directly. Every text value must be converted to a number first.

**Two encoding approaches:**

| Method | How it works | When to use |
|--------|-------------|-------------|
| `LabelEncoder` | Hot=0, Mild=1, Cool=2 | Ordered categories (Small, Medium, Large) |
| `pd.get_dummies` (One-Hot) | Creates a binary column per category | Unordered categories — Hot is not greater than Cool |

We use One-Hot Encoding because temperature categories have no natural order. Making Hot=2 and Cool=0 would imply Hot is numerically twice as large as Cool, which is meaningless.

---

## Full Pipeline

| Step | Task |
|------|------|
| 1 | Import libraries |
| 2 | Load dataset |
| 3 | EDA — 4 bar charts |
| 4 | Descriptive statistics |
| 5 | One-Hot Encoding — convert text to binary columns |
| 6 | Target detection and distribution check |
| 7 | Feature and target selection |
| 8 | Train-test split |
| 9 | Feature scaling |
| 10 | Model training |
| 11 | Predictions and evaluation |
| 12 | Predict new weather conditions |
| 13 | Batch prediction — multiple conditions |

---
## Step 1 — Import Libraries

| Library | Role |
|---------|------|
| `pandas` | Load CSV, one-hot encoding, column management |
| `numpy` | Numerical operations |
| `matplotlib.pyplot` | EDA bar charts |
| `train_test_split` | Split encoded data into training and test sets |
| `LogisticRegression` | Binary classification model |
| `accuracy_score` | Overall fraction of correct predictions |
| `confusion_matrix` | TP, TN, FP, FN breakdown |
| `classification_report` | Per-class Precision, Recall, F1 |
| `StandardScaler` | Normalize encoded features before training |

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.preprocessing import StandardScaler

---
## Step 2 — Load Dataset

We load the Golf Play dataset and immediately inspect the first few rows.

**Expected columns:**

| Column | Values | Type |
|--------|--------|------|
| `Temperature` | Hot, Mild, Cool | Categorical — 3 levels |
| `Humidity` | High, Normal | Categorical — 2 levels |
| `Windy` | Weak, Strong (or Yes/No) | Categorical — 2 levels |
| `PlayGolf` | Yes, No | Target — binary |

Inspect the column names carefully after loading — the exact string values in the dataset determine what the one-hot encoded column names will be, and mismatching these causes silent errors during new-data prediction.

In [ ]:
# step 1: Load data
df=pd.read_csv("Golf_play_datasets.txt")
print("="*60)
print("orginal data :")
print("="*60)
print(df.head())
print("="*60)
print("All unique values per column:")
for col in df.columns:
    print(f"  {col}: {df[col].unique().tolist()}")
print("="*60)

---
## Step 3 — Initial Visualization — EDA

We plot bar charts for all four columns — three features and the target. Since every column is categorical, bar charts showing category counts are the correct visualization (scatter plots would not make sense for text data).

**What to look for:**

| Plot | What to check |
|------|---------------|
| PlayGolf distribution | Is the target balanced? Severe imbalance needs special handling |
| Temperature distribution | Are all three levels represented? |
| Humidity distribution | Is one humidity level dominant? |
| Windy distribution | Is the split roughly even? |

In [ ]:
# step 2: Initail visualisation
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Plot 1: Target class distribution
play_counts = df["PlayGolf"].value_counts()
axes[0, 0].bar(play_counts.index, play_counts.values,
               color=["tomato", "steelblue"], edgecolor="white", width=0.5)
axes[0, 0].set_title("PlayGolf — Class Distribution", fontsize=12)
axes[0, 0].set_ylabel("Count")
for i, (label, val) in enumerate(play_counts.items()):
    axes[0, 0].text(i, val + 0.1, str(val), ha="center", fontsize=11)
axes[0, 0].grid(True, axis="y", linestyle="--", alpha=0.5)

# Plot 2: Temperature distribution
temp_counts = df["Temperature"].value_counts()
axes[0, 1].bar(temp_counts.index, temp_counts.values,
               color="darkorange", edgecolor="white", width=0.5)
axes[0, 1].set_title("Temperature Distribution", fontsize=12)
axes[0, 1].set_ylabel("Count")
for i, val in enumerate(temp_counts.values):
    axes[0, 1].text(i, val + 0.1, str(val), ha="center", fontsize=11)
axes[0, 1].grid(True, axis="y", linestyle="--", alpha=0.5)

# Plot 3: Humidity distribution
hum_counts = df["Humidity"].value_counts()
axes[1, 0].bar(hum_counts.index, hum_counts.values,
               color="seagreen", edgecolor="white", width=0.5)
axes[1, 0].set_title("Humidity Distribution", fontsize=12)
axes[1, 0].set_ylabel("Count")
for i, val in enumerate(hum_counts.values):
    axes[1, 0].text(i, val + 0.1, str(val), ha="center", fontsize=11)
axes[1, 0].grid(True, axis="y", linestyle="--", alpha=0.5)

# Plot 4: Windy distribution
wind_counts = df["Windy"].value_counts()
axes[1, 1].bar(wind_counts.index, wind_counts.values,
               color="mediumpurple", edgecolor="white", width=0.5)
axes[1, 1].set_title("Windy Distribution", fontsize=12)
axes[1, 1].set_ylabel("Count")
for i, val in enumerate(wind_counts.values):
    axes[1, 1].text(i, val + 0.1, str(val), ha="center", fontsize=11)
axes[1, 1].grid(True, axis="y", linestyle="--", alpha=0.5)

plt.suptitle("Exploratory Data Analysis — Feature and Target Distributions",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print("="*60)

---
## Step 4 — EDA — Descriptive Statistics

Since all columns are categorical, `df.describe()` shows character counts and top values rather than numeric stats. The important checks here are:

| Check | Why |
|-------|-----|
| `count` per column | Confirms no missing values (all should equal total rows) |
| `unique` per column | Confirms expected number of categories |
| `top` value | Most frequent category — useful for understanding class dominance |
| `df.info()` | Confirms data types — all should be `object` for categorical columns |
| `df.shape` | Total rows and columns |

In [ ]:
# step 3: EDA 
print("Basic statistic:\n",df.describe())
print("="*60)
print("Missing values:\n",df.isnull().sum())
print("="*60)
print("Shape:\n",df.shape)
print("="*60)
print("Basic info:\n",df.info())
print("="*60)
print("columns:\n",df.columns.tolist())
print("="*60)

---
## Step 5 — One-Hot Encoding

We convert every categorical column into a set of binary (0/1) columns.

**How `pd.get_dummies` works:**

For Temperature with values Hot, Mild, Cool — it creates:

| Original | Temperature_Cool | Temperature_Hot | Temperature_Mild |
|----------|-----------------|-----------------|------------------|
| Hot | 0 | 1 | 0 |
| Mild | 0 | 0 | 1 |
| Cool | 1 | 0 | 0 |

Each row gets exactly one 1 and the rest 0. This is called a one-hot vector.

**Why not just use integers (Hot=0, Mild=1, Cool=2):**  
Integers imply order and magnitude. Cool=0 and Hot=2 would suggest Hot is 2x something compared to Cool. There is no such mathematical relationship between temperature categories. One-hot encoding treats each category as independent.

**Important — we encode the target separately and drop the original text columns after encoding.**

In [ ]:
# step 4: Label encoder — One-Hot Encoding
# Create dummies and assign to new DataFrame first, then join
temp_dummies     = pd.get_dummies(df["Temperature"], prefix="Temperature")
humidity_dummies = pd.get_dummies(df["Humidity"],    prefix="Humidity")
windy_dummies    = pd.get_dummies(df["Windy"],       prefix="Windy")
playgolf_dummies = pd.get_dummies(df["PlayGolf"],    prefix="playgolf")

# Join all with original df
df = pd.concat([df, temp_dummies, humidity_dummies, windy_dummies, playgolf_dummies], axis=1)
print("Final ML ready Data:\n",df)

# Drop original text columns
df = df.drop(["Temperature", "Humidity", "Windy", "PlayGolf"], axis=1)
print("="*60)
print("After dropping original text columns:")
print(df.columns.tolist())
print("="*60)
print(df.head(10))
print("="*60)

---
## Step 6 — Target Column Detection

After one-hot encoding, `PlayGolf` becomes two columns — `playgolf_Yes` and `playgolf_No`. We use `playgolf_Yes` as the target: 1 means the person plays, 0 means they do not.

**Why dynamic detection:**  
Different datasets may have the target encoded differently depending on column names. The `if/elif` check handles this gracefully — rather than hardcoding `playgolf_Yes` and crashing if the column is named differently.

This is exactly the kind of defensive coding that separates reliable production code from fragile notebook code.

In [ ]:
# step 5: Check distribution instead of outliers
# FIXED: Look for the correct column name (playgolf_No or playgolf_Yes)
if "playgolf_Yes" in df.columns:
    target_col = "playgolf_Yes"
elif "playgolf_No" in df.columns:
    target_col = "playgolf_No"
else:
    print("Available columns:", df.columns.tolist())
    target_col = input("Enter the target column name: ")

print(f"Target column: {target_col}")
print(df[target_col].value_counts())
print("="*60)
print(f"Class balance: {df[target_col].mean()*100:.1f}% play golf")
print("="*60)

---
## Step 7 — Feature and Target Selection

We select all one-hot encoded columns as features `X`, and the detected target column as `Y`.

**Feature columns = everything except the target.**

Note: we intentionally keep both `playgolf_Yes` and `playgolf_No` would cause leakage — the model would just read `playgolf_No = 1 - playgolf_Yes` and achieve perfect accuracy trivially. That is why we drop the original PlayGolf column and only keep one of the two encoded versions as the target, using the other as part of features only if it is genuinely separate from the target.

The list comprehension `[col for col in df.columns if col != target_col]` automatically excludes only the target — this is cleaner and more robust than manually listing all feature column names.

In [ ]:
# step 6: Feature selection
# Select all dummy columns except the target
feature_cols = [col for col in df.columns if col != target_col]
X = df[feature_cols]
Y = df[target_col]

print("Feature and Target Selection")
print("="*60)
print(f"Feature columns ({len(feature_cols)}):")
for col in feature_cols:
    print(f"  {col}")
print(f"Target column  : {target_col}")
print(f"X shape        : {X.shape}")
print(f"Y shape        : {Y.shape}")
print("="*60)

---
## Step 8 — Train-Test Split

We split with `stratify=Y` to ensure both classes (Play / No Play) appear in both training and test sets. With only 14 rows in the golf dataset, random splitting without stratification could accidentally place all No Play examples in training and all Play examples in test — making evaluation meaningless.

| Parameter | Value | Effect |
|-----------|-------|--------|
| `test_size=0.2` | 20% | ~3 samples in test set |
| `random_state=42` | Fixed | Same split every run |
| `stratify=Y` | Target | Preserves class ratio in both sets |

In [ ]:
# step 6: Train test split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42,stratify=Y)

print("Train-Test Split")
print("="*60)
print(f"Total samples  : {len(X)}")
print(f"Training set   : {len(X_train)}")
print(f"Test set       : {len(X_test)}")
print("="*60)
print(f"Train class balance: {Y_train.mean()*100:.1f}% play")
print(f"Test  class balance: {Y_test.mean()*100:.1f}% play")
print("="*60)

---
## Step 9 — Feature Scaling

Even though one-hot encoded values are already 0 or 1, StandardScaler is still applied for consistency. In this dataset it changes values slightly because the mean of each binary column is not necessarily 0.5, so subtracting the mean and dividing by std centers them.

**The fit-on-train-only rule always applies:**

| Call | Data | Why |
|------|------|-----|
| `fit_transform(X_train)` | Training only | Learns mean and std from training data |
| `transform(X_test)` | Test only | Applies same transformation — no new learning |

In [ ]:
# step 7: Feature scaling
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

print("StandardScaler Applied")
print("="*60)
print("X_train_scaled — first 3 rows:")
print(pd.DataFrame(X_train_scaled, columns=feature_cols).head(3).round(4).to_string())
print("="*60)

---
## Step 10 — Model Training

We train `LogisticRegression` on the scaled training data. After training, we print the learned coefficients — one per feature column.

Because features are scaled, coefficients are directly comparable:
- Large positive coefficient = that weather condition strongly increases the probability of playing
- Large negative coefficient = that condition strongly decreases the probability of playing
- Near zero = that condition has little influence on the decision

In [ ]:
# step 8: Model selction 
model=LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled,Y_train)

print("Model Trained")
print("="*60)
print(f"Intercept : {model.intercept_[0]:.4f}")
print("Learned Coefficients:")
coef_df = pd.DataFrame({
    "Feature"     : feature_cols,
    "Coefficient" : model.coef_[0].round(4)
}).sort_values("Coefficient", ascending=False)
print(coef_df.to_string(index=False))
print("="*60)

---
## Step 11 — Predictions and Evaluation

We generate predictions and evaluate using three standard classification tools:

| Tool | What it shows |
|------|---------------|
| `accuracy_score(Y_test, y_pred)` | Overall % correct |
| `confusion_matrix(Y_test, y_pred)` | TP, TN, FP, FN counts |
| `classification_report(Y_test, y_pred)` | Per-class Precision, Recall, F1 |

**Note on argument order — a common mistake:**  
`accuracy_score(y_true, y_pred)` — actual labels come first, predictions second. Swapping them can give wrong results silently for imbalanced datasets.

**Reading the classification report:**
- `0` = No Play, `1` = Play
- `support` = number of actual samples of that class in the test set
- With only 3 test samples, every metric should be interpreted with caution — the dataset is too small for reliable generalization estimates

In [ ]:
# step 9: Predictions
y_pred=model.predict(X_test_scaled)
y_prob=model.predict_proba(X_test_scaled)

# step 10: Evalution
# FIXED: correct argument order — (y_true, y_pred)
print("accuracy:",accuracy_score(Y_test,y_pred))
print("confusion matrix:\n",confusion_matrix(Y_test,y_pred))
print("Classification report:\n",classification_report(Y_test,y_pred))
print("="*60)

# Prediction table — actual vs predicted
result_df = pd.DataFrame({
    "Actual"    : Y_test.values,
    "Predicted" : y_pred,
    "P(No Play)": y_prob[:,0].round(3),
    "P(Play)"   : y_prob[:,1].round(3),
    "Correct"   : (Y_test.values == y_pred).astype(int)
})
print("Test predictions:")
print(result_df.to_string(index=False))
print("="*60)

---
## Step 12 — Confusion Matrix Visualization

We plot the confusion matrix as a colored heatmap and show the predicted probabilities for each test sample.

In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(Y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im = axes[0].imshow(cm, cmap="Blues")
plt.colorbar(im, ax=axes[0])
cell_labels = [["TN", "FP"], ["FN", "TP"]]
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, f"{cell_labels[i][j]}\n{cm[i,j]}",
                     ha="center", va="center", fontsize=13, fontweight="bold",
                     color="white" if cm[i,j] > cm.max()/2 else "black")
axes[0].set_xticks([0,1])
axes[0].set_yticks([0,1])
axes[0].set_xticklabels(["Predicted No Play","Predicted Play"], fontsize=9)
axes[0].set_yticklabels(["Actual No Play","Actual Play"], fontsize=9)
axes[0].set_title("Confusion Matrix", fontsize=12)

# Probability bars
bar_colors = ["steelblue" if y==1 else "tomato" for y in Y_test.values]
bars = axes[1].bar(range(len(y_prob[:,1])), y_prob[:,1],
                   color=bar_colors, edgecolor="white", width=0.5)
axes[1].axhline(0.5, color="black", linestyle="--", linewidth=1.5,
                label="Decision boundary (0.5)")
for i,(bar,prob) in enumerate(zip(bars, y_prob[:,1])):
    axes[1].text(i, prob+0.03, f"{prob:.2f}", ha="center", fontsize=10)
axes[1].set_xticks(range(len(y_prob[:,1])))
axes[1].set_xticklabels([f"S{i+1}" for i in range(len(y_prob[:,1]))], fontsize=10)
axes[1].set_ylabel("P(Play)")
axes[1].set_ylim(0, 1.2)
axes[1].set_title("Predicted Play Probabilities\n(Blue=Actual Play, Red=Actual No Play)")
axes[1].legend(fontsize=9)
axes[1].grid(True, axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()
print("="*60)

---
## Step 13 — Predict New Weather Condition

We predict for a brand new weather condition — Hot, High Humidity, Weak Wind.

**The critical challenge with new data and one-hot encoding:**  
When we call `pd.get_dummies` on a single row with Temperature=Hot, it only creates the column `Temperature_Hot`. It does not create `Temperature_Cool` or `Temperature_Mild` because those values are not present in that single row. But the model was trained on all three columns — feeding a DataFrame with missing columns would crash or silently produce wrong results.

**The fix:**  
After encoding the new row, loop through all training feature columns. If any is missing from the new row, add it with value 0. Then reorder columns to match training exactly.

This is one of the most common production bugs in ML pipelines.

In [ ]:
# step 11: predictiong new values
new_data=pd.DataFrame({
    "Temperature":["Hot"],
    "Humidity":["High"],
    "Windy":["Weak"]     # FIXED: was 'Week' (typo) — corrected to 'Weak'
})
# Encode the data same way
new_temp    =pd.get_dummies(new_data["Temperature"],prefix="Temperature")
new_humidity=pd.get_dummies(new_data["Humidity"],   prefix="Humidity")
new_windy   =pd.get_dummies(new_data["Windy"],      prefix="Windy")

# Combine encoded features — drop original text columns
new_encoded=pd.concat([new_temp,new_humidity,new_windy],axis=1)

# FIXED: Add missing columns that are in training data but not in new data
for col in feature_cols:
    if col not in new_encoded.columns:
        new_encoded[col] = 0

# Reorder columns to match training data exactly
new_encoded=new_encoded[feature_cols]

# step 13: scale and predict
new_scaled=scaler.transform(new_encoded)
prediction=model.predict(new_scaled)
probability=model.predict_proba(new_scaled)

print(f"Prediction for Hot, High, Weak: {'Play' if prediction[0]==1 else 'No Play'}")
print(f"Probability: {probability[0][1]*100:.2f}% chance of playing")
print("="*60)

---
## Step 14 — Batch Prediction — Multiple Weather Conditions

We loop through multiple weather scenarios and print the model's decision for each. This is exactly how a deployed model would work in a real application — given any new input, apply the same encoding pipeline and get a prediction.

**Why this pattern matters:**  
The encoding + missing column fill + reorder pattern must be applied identically for every new input. This is why production ML systems wrap all of this into a single function or Scikit-Learn `Pipeline` object — so the same steps are guaranteed to run in the same order every time.

In [ ]:
# optional work: Try predicting different conditions
test_conditions = [
    ['Hot',  'High',   'Strong'],
    ['Cool', 'Normal', 'Weak'],
    ['Mild', 'High',   'Strong']
]

print("Batch Predictions")
print("="*60)

for temp, humid, wind in test_conditions:
    # Create new data
    new_data = pd.DataFrame({
        'Temperature': [temp],
        'Humidity':    [humid],
        'Windy':       [wind]
    })
    
    # Encode (same process as before)
    new_temp     = pd.get_dummies(new_data['Temperature'], prefix="Temperature")
    new_humidity = pd.get_dummies(new_data['Humidity'],    prefix="Humidity")
    new_windy    = pd.get_dummies(new_data['Windy'],       prefix="Windy")
    
    new_encoded = pd.concat([new_temp, new_humidity, new_windy], axis=1)
    
    # Add missing columns
    for col in feature_cols:
        if col not in new_encoded.columns:
            new_encoded[col] = 0
    
    new_encoded = new_encoded[feature_cols]
    new_scaled  = scaler.transform(new_encoded)
    
    pred = model.predict(new_scaled)[0]
    prob = model.predict_proba(new_scaled)[0][1]
    
    print(f"{temp:6s}, {humid:7s}, {wind:7s}  ->  {'Play   ' if pred==1 else 'No Play'}  ({prob:.1%} chance)")

print("="*60)

---
## Step 15 — Feature Importance Visualization

The learned coefficients tell us which weather conditions most strongly influence the play/no-play decision. We plot them as a horizontal bar chart.

In [ ]:
# Feature importance — learned coefficients
coefs   = model.coef_[0]
colors  = ["steelblue" if c > 0 else "tomato" for c in coefs]

plt.figure(figsize=(8, 5))
bars = plt.barh(feature_cols, coefs, color=colors, edgecolor="white", height=0.5)
plt.axvline(0, color="black", linewidth=0.8, linestyle="--")
for bar, coef in zip(bars, coefs):
    offset = 0.05 if coef > 0 else -0.05
    plt.text(coef + offset,
             bar.get_y() + bar.get_height()/2,
             f"{coef:+.3f}",
             va="center",
             ha="left" if coef > 0 else "right",
             fontsize=9)
plt.xlabel("Coefficient (log-odds contribution)")
plt.title("Feature Importance — Logistic Regression Weights\n(Blue = increases play probability, Red = decreases)")
plt.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
print("="*60)

---
## Pipeline Summary

| Step | Action | Key concept |
|------|--------|-------------|
| 1 | Import all libraries | StandardScaler, LogisticRegression, metrics |
| 2 | Load Golf_play_datasets.txt | Print unique values per column immediately |
| 3 | EDA — 2x2 bar chart grid | Categorical data needs bar charts not scatter |
| 4 | describe(), isnull(), info(), shape | Full data quality check |
| 5 | pd.get_dummies on all 4 columns | One-hot encodes text to binary columns |
| 6 | Dynamic target column detection | if/elif defensive check for column names |
| 7 | feature_cols = all except target | List comprehension — clean and robust |
| 8 | train_test_split with stratify=Y | Preserves class ratio in both sets |
| 9 | StandardScaler — fit on train only | Same rule as always |
| 10 | LogisticRegression().fit() | Print coefficients after training |
| 11 | predict() + predict_proba() | accuracy, confusion matrix, classification report |
| 12 | Confusion matrix heatmap + probability bars | Visual model summary |
| 13 | Single new condition prediction | Encode + fill missing + reorder + scale + predict |
| 14 | Batch prediction loop | Same pipeline applied to 3 conditions |
| 15 | Feature importance bar chart | Which weather condition matters most |

---

## Bugs Fixed from Original Code

| Bug | Original | Fixed |
|-----|----------|-------|
| Argument order | `accuracy_score(y_pred, Y_test)` | `accuracy_score(Y_test, y_pred)` |
| Typo in new data | `"Windy": ["Week"]` | `"Windy": ["Weak"]` |
| Text columns in new_encoded | Original text columns included before reorder | Dropped — only dummy columns passed |
| No stratification | `train_test_split(X, Y, test_size=0.2)` | Added `stratify=Y` |
| max_iter warning | Default 100 iterations | `max_iter=1000` to ensure convergence |

---

**Next notebook:** `03_knn_classification.ipynb` — K-Nearest Neighbors on a real dataset